# Lakekeeper + Lance + Spark

End-to-end example using the **Java client** (`lakekeeper-client-spark`) from PySpark via py4j:

1. Start Spark with the Lakekeeper JAR on the classpath
2. Build a `LakekeeperClient` through `spark._jvm`
3. Create a Lance generic table via the Java API
4. Write an embeddings dataset with `pylance` (fast Python writer)
5. Read it back with `LakekeeperSpark.read()` — credential wiring handled by the Java shim

## Requirements

```
pip install pylance pyarrow numpy pyspark
```

Download JARs and set the env vars below:
- `lakekeeper-client-spark-3.5_2.13-0.1.0.jar` from GitHub Packages
- Lance Spark connector JAR from https://github.com/lancedb/lance/releases

## Environment variables

| Variable | Default | Description |
|---|---|---|
| `LAKEKEEPER` | `http://localhost:8181` | Lakekeeper base URL |
| `WAREHOUSE_ID` | _(required)_ | Warehouse UUID or name |
| `TOKEN` | — | Static bearer token |
| `OAUTH_TOKEN_URL` / `OAUTH_CLIENT_ID` / `OAUTH_CLIENT_SECRET` / `OAUTH_SCOPE` | — | OAuth2 alternative |
| `LAKEKEEPER_JAR` | _(required)_ | Path to `lakekeeper-client-spark-*.jar` |
| `LANCE_SPARK_JAR` | — | Path to Lance Spark connector JAR |

In [ ]:
import os

LAKEKEEPER     = os.environ.get('LAKEKEEPER', 'http://localhost:8181')
WAREHOUSE_ID   = os.environ['WAREHOUSE_ID']
PROJECT_ID     = os.environ.get('PROJECT_ID')
NAMESPACE      = os.environ.get('NAMESPACE', 'examples')
TABLE          = os.environ.get('TABLE', 'spark_lance_embeddings')
EMBED_DIM      = int(os.environ.get('EMBEDDING_DIM', '128'))
ROWS           = int(os.environ.get('ROWS', '1000'))
LAKEKEEPER_JAR = os.environ['LAKEKEEPER_JAR']
LANCE_JAR      = os.environ.get('LANCE_SPARK_JAR', '')

## 1. Start PySpark with Lakekeeper + Lance JARs

In [ ]:
from pyspark.sql import SparkSession, DataFrame

jars = ','.join(j for j in [LAKEKEEPER_JAR, LANCE_JAR] if j)

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('lakekeeper-lance-demo')
    .config('spark.jars', jars)
    .config('spark.sql.shuffle.partitions', '4')
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate()
)
spark

## 2. Build the Java `LakekeeperClient` via py4j

`spark._jvm` gives direct access to any class on the Spark JVM classpath.

In [ ]:
jvm = spark._jvm

# Auth — static token or OAuth2 client_credentials
if token := os.environ.get('TOKEN'):
    auth = jvm.io.lakekeeper.client.auth.StaticToken(token)
else:
    auth = jvm.io.lakekeeper.client.auth.ClientCredentials(
        os.environ['OAUTH_TOKEN_URL'],
        os.environ['OAUTH_CLIENT_ID'],
        os.environ['OAUTH_CLIENT_SECRET'],
        os.environ.get('OAUTH_SCOPE'),   # scope (nullable)
        60,                              # refreshMarginSeconds
        30,                              # timeoutSeconds
    )

builder = (
    jvm.io.lakekeeper.client.LakekeeperClient.builder()
    .baseUrl(LAKEKEEPER)
    .warehouse(WAREHOUSE_ID)
    .auth(auth)
)
if PROJECT_ID:
    builder = builder.projectId(PROJECT_ID)

client = builder.build()
print('Java LakekeeperClient ready')

## 3. Create the Lance generic table

In [ ]:
GenericTableFormat = jvm.io.lakekeeper.client.GenericTableFormat
ConflictException  = jvm.io.lakekeeper.client.exception.ConflictException

try:
    resp = client.genericTables().create(
        NAMESPACE, TABLE,
        GenericTableFormat.normalize(GenericTableFormat.LANCE),
        None,                                    # base_location — let server assign
        None,                                    # doc
        {'embedding-dim': str(EMBED_DIM)},       # properties (Java Map via py4j)
    )
    print(f'created  → {resp.getLocation()}')
except Exception as e:
    if 'ConflictException' in type(e).__name__ or '409' in str(e):
        print(f'table {NAMESPACE}.{TABLE} already exists — continuing')
    else:
        raise

## 4. Resolve location + vended credentials for the Lance write

Call `load(vended=True)` via the Java client. The `storage-credentials` come back as
Iceberg-style S3 keys; translate them to Lance `storage_options` for the Python write.

In [ ]:
ICEBERG_TO_LANCE = {
    's3.access-key-id':     'aws_access_key_id',
    's3.secret-access-key': 'aws_secret_access_key',
    's3.session-token':     'aws_session_token',
    's3.region':            'aws_region',
    'client.region':        'aws_region',
    's3.endpoint':          'aws_endpoint',
}

def vended_response_to_lance_opts(java_resp) -> dict:
    opts = {}
    for cred in java_resp.getStorageCredentials():
        for k, v in cred.getConfig().items():
            lance_key = ICEBERG_TO_LANCE.get(k)
            if lance_key:
                opts[lance_key] = v
    config = java_resp.getConfig()
    if config:
        for k, v in config.items():
            lance_key = ICEBERG_TO_LANCE.get(k)
            if lance_key:
                opts[lance_key] = v
    endpoint = opts.get('aws_endpoint', '')
    if endpoint.startswith('http://'):
        opts['allow_http'] = 'true'
    return opts

# Fresh vended creds for the write
java_resp = client.genericTables().load(NAMESPACE, TABLE, True)
location  = java_resp.getLocation()
lance_opts = vended_response_to_lance_opts(java_resp)
print(f'location = {location}')

## 5. Write with `pylance`

In [ ]:
import numpy as np
import pyarrow as pa
import lance

def make_data(rows: int, embed_dim: int) -> pa.Table:
    rng = np.random.default_rng(42)
    embeddings = rng.standard_normal((rows, embed_dim)).astype(np.float32)
    return pa.table({
        'id':        pa.array(range(rows), type=pa.int64()),
        'label':     pa.array([f'item-{i:05d}' for i in range(rows)]),
        'embedding': pa.FixedSizeListArray.from_arrays(
                         pa.array(embeddings.reshape(-1), type=pa.float32()),
                         embed_dim,
                     ),
    })

lance.write_dataset(
    make_data(ROWS, EMBED_DIM),
    location,
    storage_options=lance_opts,
    mode='overwrite',
)
print(f'wrote {ROWS} rows to {location}')

## 6. Read back with `LakekeeperSpark.read()` (Java shim)

`LakekeeperSpark.read()` calls Lakekeeper, translates credentials into `fs.s3a.*`
Hadoop config, and returns a `Dataset<Row>`. Wrap it as a Python `DataFrame`.

In [ ]:
LakekeeperSpark = jvm.io.lakekeeper.spark.LakekeeperSpark

java_df = LakekeeperSpark.read(spark._jsparkSession, client, NAMESPACE, TABLE)
df = DataFrame(java_df, spark)

print(f'row count: {df.count()}')
df.printSchema()

## 7. Explore

In [ ]:
df.select('id', 'label').show(10)

In [ ]:
df.createOrReplaceTempView('embeddings')
spark.sql('''
    SELECT label, size(embedding) AS dim
    FROM   embeddings
    LIMIT  5
''').show()

## 8. Write back from Spark via the Java shim

`LakekeeperSpark.write()` fetches fresh vended credentials before saving.

In [ ]:
filtered = df.filter('id < 100')

# Pass a java.util.HashMap as writeOptions
opts = jvm.java.util.HashMap()
opts.put('mode', 'overwrite')

LakekeeperSpark.write(spark._jsparkSession, client, NAMESPACE, TABLE, filtered._jdf, opts)
print('wrote 100-row subset back')

## 9. Cleanup

In [ ]:
# Uncomment to drop:
# client.genericTables().drop(NAMESPACE, TABLE)
# print('table dropped')

client.close()
spark.stop()